# 6.15 — Batch Normalization

Batch normalization (BatchNorm) stabilizes deep-network training by standardizing a layer's activations with minibatch statistics, then learning a scale and shift so the network can restore any useful magnitude. In this lesson, you will build the forward pass, the learned affine restoration, training-versus-inference behavior, and the practical stability checks from scratch with NumPy.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build batch normalization one idea at a time. Run each cell in order and read the printed intermediate values — every statistic, normalized activation, affine parameter, and running estimate is visible. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, axis reductions, and numerical checks for BatchNorm.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for toy minibatches.

### 1. Minibatch statistics: estimate the local scale

BatchNorm begins with a minibatch of pre-activation values. For a dense layer, each row is one example and each column is one feature channel, so the batch mean and variance are computed **down the batch axis** for every feature separately. Those statistics are local estimates: they are not universal truths about the data distribution, but they are good enough to keep the next layer's input scale predictable during one training step.

In [ ]:
X_w = np.array([[2.0, 10.0, -1.0],
                [4.0, 12.0,  1.0],
                [6.0, 14.0,  3.0],
                [8.0, 16.0,  5.0]])  # 4 examples x 3 feature channels.
mu_w = X_w.mean(axis=0)  # one mean per feature, averaged over examples.
var_w = X_w.var(axis=0)  # one population variance per feature, averaged over examples.
print("batch mean:", mu_w)
print("batch variance:", var_w)
assert np.allclose(mu_w, [5.0, 13.0, 2.0])
assert np.allclose(var_w, [5.0, 5.0, 5.0])

▶ What you'll see: every feature has a different mean but the same variance in this tidy toy batch.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["f0", "f1", "f2"], mu_w, color="steelblue", label="mean")
plt.errorbar(["f0", "f1", "f2"], mu_w, yerr=np.sqrt(var_w), fmt="none", ecolor="black", capsize=5, label="± std")
plt.title("1: batch mean and scale by feature")
plt.ylabel("activation value")
plt.legend()
plt.show()

▶ What you'll see: feature 1 is centered much higher than feature 0 or 2, even though their spreads match.

*Why it's done this way:* downstream layers see features column-by-column, so the normalization must remove each feature's own offset and scale. Averaging over the batch gives a cheap stochastic estimate of the full-data statistics, matching the minibatch logic already used by SGD.

### 2. Normalize: subtract the mean and divide by the standard deviation

The normalized activation is

$$\hat x_{ij}=\frac{x_{ij}-\mu_j}{\sqrt{\sigma_j^2+\epsilon}}.$$

Subtracting the mean recenters each feature at zero; dividing by the standard deviation expresses every value in "how many batch standard deviations from typical." The small $\epsilon$ is not decoration — it prevents division by zero and makes the formula stable when a feature is nearly constant.

In [ ]:
eps_w = 1e-5  # tiny positive constant for numerical safety.
Xhat_w = (X_w - mu_w) / np.sqrt(var_w + eps_w)  # standardize each feature column.
print("normalized means:", np.round(Xhat_w.mean(axis=0), 6))
print("normalized variances:", np.round(Xhat_w.var(axis=0), 6))
assert np.allclose(Xhat_w.mean(axis=0), np.zeros(3), atol=1e-12)
assert np.allclose(np.round(Xhat_w.var(axis=0), 6), [0.999998, 0.999998, 0.999998])

▶ What you'll see: the normalized batch has mean 0 and variance almost 1 for every feature.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(X_w[:, 1], marker="o", label="raw feature 1")
plt.plot(Xhat_w[:, 1], marker="o", label="normalized feature 1")
plt.axhline(0, color="black", linewidth=0.7)
plt.title("2: raw scale becomes standard-deviation units")
plt.xlabel("example index")
plt.legend()
plt.show()

▶ What you'll see: raw values around 10–16 become symmetric standardized values around −1.34 to +1.34.

*Why it's done this way:* subtracting $\mu$ removes an additive shift and dividing by $\sqrt{\sigma^2+\epsilon}$ removes multiplicative scale. That makes the next affine layer less sensitive to drifting activation magnitudes, while $\epsilon$ keeps the denominator mathematically legal.

### 3. Learned scale and shift: restore what normalization should not destroy

If BatchNorm always forced every feature to mean 0 and variance 1, it could erase useful representation choices. The learned parameters $\gamma$ and $\beta$ fix that by applying an affine map after normalization:

$$y_{ij}=\gamma_j\hat x_{ij}+\beta_j.$$

When $\gamma=1$ and $\beta=0$, BatchNorm is pure standardization. During training, the network learns whether each feature should be amplified, damped, shifted positive, or shifted negative.

In [ ]:
gamma_w = np.array([1.5, 0.5, 2.0])  # learned scale per feature.
beta_w = np.array([0.0, 3.0, -1.0])  # learned shift per feature.
Y_w = gamma_w * Xhat_w + beta_w  # broadcast affine restoration across the batch.
print("output means:", np.round(Y_w.mean(axis=0), 3))
print("output variances:", np.round(Y_w.var(axis=0), 3))
assert np.allclose(np.round(Y_w.mean(axis=0), 3), [0.0, 3.0, -1.0])
assert np.allclose(np.round(Y_w.var(axis=0), 3), [2.25, 0.25, 4.0], atol=1e-3)

▶ What you'll see: output means match beta, and output variances are approximately gamma squared.

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(Xhat_w[:, 0], Y_w[:, 0], color="seagreen", label="feature 0")
plt.plot(Xhat_w[:, 0], gamma_w[0] * Xhat_w[:, 0] + beta_w[0], color="black", linestyle="--")
plt.title("3: γ stretches and β shifts normalized values")
plt.xlabel("x_hat")
plt.ylabel("y")
plt.legend()
plt.show()

▶ What you'll see: feature 0 lies on a straight line with slope γ=1.5 and intercept β=0.

*Why it's done this way:* standardization is a constraint, and $\gamma,\beta$ are the escape hatch. The model gets stable intermediate statistics but does not lose expressive power, because it can learn the exact scale and offset that the following nonlinearity or layer needs.

### 4. Training versus inference: minibatch stats become running stats

During training, BatchNorm uses the current minibatch statistics. During inference, a single example or tiny batch would give noisy statistics, so the layer uses running estimates accumulated during training. A common update is

$$\mu_{run}\leftarrow m\mu_{run}+(1-m)\mu_B,$$

and the same form is used for variance. Here $m$ is momentum: high momentum trusts history, low momentum reacts quickly.

In [ ]:
batch_means_w = np.array([[5.0, 13.0, 2.0],
                          [6.0, 12.0, 1.0],
                          [4.0, 14.0, 3.0]])  # three minibatch means seen during training.
running_w = np.zeros(3)  # start from a neutral running mean.
momentum_w = 0.8  # keep 80% history, add 20% current batch.
history_w = []
for mean_w in batch_means_w:
    running_w = momentum_w * running_w + (1 - momentum_w) * mean_w
    history_w.append(running_w.copy())
print("running means after each batch:\n", np.round(history_w, 3))
assert np.allclose(np.round(history_w[-1], 3), [2.4, 6.384, 1.016])

▶ What you'll see: the running estimate moves toward the batch means gradually instead of jumping all the way.

In [ ]:
x_single_w = np.array([[7.0, 15.0, 4.0]])  # one inference example.
running_var_w = np.array([5.0, 5.0, 5.0])  # pretend variance estimate learned during training.
y_infer_w = gamma_w * ((x_single_w - running_w) / np.sqrt(running_var_w + eps_w)) + beta_w
print("inference output:", np.round(y_infer_w, 3))
assert y_infer_w.shape == (1, 3)

▶ What you'll see: one example can be normalized without asking that one example to define its own variance.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(np.array(history_w)[:, 0], marker="o", label="running mean f0")
plt.axhline(batch_means_w[:, 0].mean(), color="gray", linestyle="--", label="average batch mean")
plt.title("4: running statistics smooth minibatch noise")
plt.xlabel("training batch")
plt.ylabel("mean estimate")
plt.legend()
plt.show()

▶ What you'll see: the running mean follows the batches smoothly rather than matching each noisy batch exactly.

*Why it's done this way:* training can exploit minibatch statistics because each update already sees a batch, but inference needs deterministic behavior. Running averages transfer the training-time notion of typical scale into deployment.

### 5. The optimization effect: gradients see a better-conditioned scale

BatchNorm does not merely make activations look tidy; it changes the scale seen by downstream parameters. If one feature has values around 100 and another around 0.1, a shared learning rate produces very different gradient magnitudes. Standardizing features makes the curvature more balanced, so one step size is less likely to be too small for one coordinate and too large for another.

In [ ]:
X_bad_w = np.column_stack([np.linspace(90, 110, 6), np.linspace(-0.2, 0.2, 6)])  # two wildly different scales.
target_w = np.linspace(0, 1, 6)  # simple target for a linear-output toy loss.
w_w = np.array([0.01, 0.01])  # same starting weight for both features.
grad_bad_w = (2 / len(target_w)) * X_bad_w.T @ (X_bad_w @ w_w - target_w)  # MSE gradient.
X_good_w = (X_bad_w - X_bad_w.mean(axis=0)) / np.sqrt(X_bad_w.var(axis=0) + eps_w)  # BatchNorm-style inputs.
grad_good_w = (2 / len(target_w)) * X_good_w.T @ (X_good_w @ w_w - target_w)  # gradient after standardization.
print("raw gradient:", np.round(grad_bad_w, 3))
print("normalized gradient:", np.round(grad_good_w, 3))
assert abs(grad_bad_w[0]) > 1000 * abs(grad_bad_w[1])
assert abs(grad_good_w[0]) < 2.0

▶ What you'll see: the raw feature creates a huge gradient, while normalized inputs keep gradients comparable.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["raw f0", "raw f1", "norm f0", "norm f1"],
        [abs(grad_bad_w[0]), abs(grad_bad_w[1]), abs(grad_good_w[0]), abs(grad_good_w[1])],
        color=["crimson", "crimson", "seagreen", "seagreen"])
plt.yscale("log")
plt.title("5: normalization balances gradient scale")
plt.ylabel("absolute gradient, log scale")
plt.show()

▶ What you'll see: the raw gradients differ by orders of magnitude, while normalized gradients are on a shared scale.

*Why it's done this way:* gradient descent uses one learning rate to update many parameters. By keeping activations near zero mean and unit variance, BatchNorm makes that single learning rate more reasonable across coordinates, which is a major part of its training-stability benefit.

### 6. Small-batch and epsilon pitfalls: the statistics can lie

BatchNorm's statistics are only as good as the minibatch. Very small batches produce noisy means and variances; a constant feature produces variance zero. The $\epsilon$ term prevents numerical failure, but it cannot invent trustworthy statistics. This is why BatchNorm works best when the batch axis contains enough examples and why alternatives like layer normalization matter for tiny-batch regimes.

In [ ]:
constant_w = np.array([[3.0], [3.0], [3.0]])  # zero-variance feature.
mu_const_w = constant_w.mean(axis=0)
var_const_w = constant_w.var(axis=0)
xhat_const_w = (constant_w - mu_const_w) / np.sqrt(var_const_w + eps_w)
print("variance:", var_const_w)
print("normalized constant feature:", xhat_const_w.ravel())
assert np.allclose(xhat_const_w, 0.0)

▶ What you'll see: epsilon prevents division by zero, and all centered constant values become 0.

In [ ]:
rng_w = np.random.default_rng(0)
big_batch_w = rng_w.normal(loc=2.0, scale=3.0, size=128)
small_batch_w = big_batch_w[:2]
print("large-batch mean/var:", round(big_batch_w.mean(), 3), round(big_batch_w.var(), 3))
print("tiny-batch mean/var:", round(small_batch_w.mean(), 3), round(small_batch_w.var(), 3))
assert small_batch_w.size == 2

▶ What you'll see: the two-example batch gives a much noisier estimate than the 128-example batch.

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(big_batch_w, bins=16, alpha=0.7, label="128 examples")
plt.scatter(small_batch_w, np.zeros_like(small_batch_w), color="red", s=80, label="2-example batch")
plt.title("6: tiny batches estimate distribution poorly")
plt.xlabel("activation value")
plt.legend()
plt.show()

▶ What you'll see: two red points cannot summarize the whole activation distribution reliably.

*Why it's done this way:* $\epsilon$ solves the algebraic singularity, not the statistical problem. BatchNorm deliberately uses a batch estimate for efficiency and regularization, but the variance of that estimate is a real modeling constraint.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, reductions, broadcasting, and reproducible toy training loops.
import matplotlib.pyplot as plt # load Matplotlib for the histograms, heatmaps, and diagnostic plots.
np.random.seed(0) # make every random example deterministic.

def batch_norm_forward(x, gamma, beta, eps=1e-5): # implement training-time BatchNorm for dense activations.
    mu = x.mean(axis=0) # compute one batch mean per feature.
    var = x.var(axis=0) # compute one batch variance per feature.
    xhat = (x - mu) / np.sqrt(var + eps) # standardize feature columns safely.
    y = gamma * xhat + beta # restore learned scale and shift.
    return y, xhat, mu, var # return all pieces so examples can inspect the math.

def running_update(running, batch_value, momentum=0.9): # update a running statistic with exponential smoothing.
    return momentum * running + (1 - momentum) * batch_value # keep history and add a fraction of the current batch.

def show_feature_stats(values, title): # compact helper for visualizing feature means and standard deviations.
    means = values.mean(axis=0) # one mean per feature.
    stds = values.std(axis=0) # one standard deviation per feature.
    plt.figure(figsize=(4.5, 3)) # create a compact figure.
    plt.bar(np.arange(values.shape[1]), means, yerr=stds, capsize=4, color="steelblue") # draw mean ± std per feature.
    plt.title(title) # label the diagnostic plot.
    plt.xlabel("feature") # label feature axis.
    plt.ylabel("mean ± std") # label statistic axis.
    plt.show() # display the chart.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.
## 🟢 Basics (warm-up)

### Basic 1 — Compute batch mean

**Goal.** Compute per-feature means, because BatchNorm removes each feature's batch offset before measuring scale. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[1., 10.], [3., 14.], [5., 18.]]) # create three examples with two differently centered features.
mu_b1 = X_b1.mean(axis=0) # average down rows so each feature gets its own mean.
print("batch means:", mu_b1) # inspect the offsets BatchNorm will subtract.
assert np.allclose(mu_b1, [3., 14.]) # verify the concrete means.

▶ What you'll see: feature 0 is centered at 3, while feature 1 is centered at 14.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact mean plot.
plt.bar(["f0", "f1"], mu_b1, color="teal") # show the two feature offsets.
plt.title("Basic 1: per-feature batch means") # title the figure.
plt.ylabel("mean") # label the mean scale.
plt.show() # display the chart.

▶ What you'll see: the second feature has a much larger offset even in the same batch.

👀 Takeaway: BatchNorm computes statistics per feature, not one global statistic for the whole matrix.

### Basic 2 — Compute batch variance

**Goal.** Compute per-feature variance, because BatchNorm divides by feature-wise spread rather than by raw magnitude. We build it in 2 steps.

In [ ]:
X_b2 = np.array([[1., 10.], [3., 14.], [5., 18.]]) # reuse the simple two-feature batch.
var_b2 = X_b2.var(axis=0) # compute population variance along the batch axis.
std_b2 = np.sqrt(var_b2) # take square roots to get standard deviations.
print("variance:", np.round(var_b2, 3), "std:", np.round(std_b2, 3)) # inspect spread statistics.
assert np.allclose(np.round(var_b2, 3), [2.667, 10.667]) # verify the concrete variances.

▶ What you'll see: feature 1 has four times the variance of feature 0.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact variance chart.
plt.bar(["f0", "f1"], var_b2, color="orange") # compare per-feature variances.
plt.title("Basic 2: per-feature batch variances") # title the figure.
plt.ylabel("variance") # label the variance scale.
plt.show() # display the chart.

▶ What you'll see: variance exposes scale differences even after ignoring the mean.

👀 Takeaway: centering alone is not enough; BatchNorm also normalizes feature spread.

### Basic 3 — Center activations

**Goal.** Subtract the batch mean, because centering makes each feature's average activation zero. We build it in 2 steps.

In [ ]:
X_b3 = np.array([[1., 10.], [3., 14.], [5., 18.]]) # define a small activation batch.
centered_b3 = X_b3 - X_b3.mean(axis=0) # subtract each feature mean by broadcasting.
print("centered batch:\n", centered_b3) # inspect deviations from the feature means.
assert np.allclose(centered_b3.mean(axis=0), [0., 0.]) # verify zero-centered features.

▶ What you'll see: each column now has negative, zero, and positive deviations around 0.

In [ ]:
plt.figure(figsize=(4, 3)) # create a centered-value plot.
plt.plot(centered_b3[:, 0], marker="o", label="f0 centered") # show feature 0 deviations.
plt.plot(centered_b3[:, 1], marker="o", label="f1 centered") # show feature 1 deviations.
plt.axhline(0, color="black", linewidth=0.8) # mark the zero baseline.
plt.title("Basic 3: subtracting μ centers features") # title the figure.
plt.legend() # show feature labels.
plt.show() # display the chart.

▶ What you'll see: both features cross the zero line after mean subtraction.

👀 Takeaway: subtracting μ removes additive drift from the layer's activations.

### Basic 4 — Normalize to unit variance

**Goal.** Divide centered activations by standard deviation, because normalized units put features on comparable scales. We build it in 2 steps.

In [ ]:
X_b4 = np.array([[1., 10.], [3., 14.], [5., 18.]]) # define a small activation batch.
eps_b4 = 1e-5 # add numerical safety to the denominator.
Xhat_b4 = (X_b4 - X_b4.mean(axis=0)) / np.sqrt(X_b4.var(axis=0) + eps_b4) # compute normalized activations.
print("xhat:\n", np.round(Xhat_b4, 3)) # inspect standard-deviation units.
assert np.allclose(np.round(Xhat_b4.var(axis=0), 6), [0.999996, 0.999999]) # verify near-unit variance.

▶ What you'll see: both columns now have almost the same standardized spread.

In [ ]:
show_feature_stats(Xhat_b4, "Basic 4: normalized mean ± std") # visualize normalized feature statistics.

▶ What you'll see: bars sit near 0 with error bars near 1.

👀 Takeaway: BatchNorm changes raw activations into standardized z-score-like activations.

### Basic 5 — Use epsilon for safety

**Goal.** Normalize a nearly constant feature, because epsilon prevents division by zero when variance is tiny. We build it in 2 steps.

In [ ]:
X_b5 = np.array([[2.0], [2.0], [2.0], [2.0]]) # create a constant feature with zero variance.
var_b5 = X_b5.var(axis=0) # compute the exact zero variance.
den_b5 = np.sqrt(var_b5 + 1e-5) # add epsilon before taking the square root.
print("variance:", var_b5, "safe denominator:", np.round(den_b5, 6)) # inspect the protected denominator.
assert var_b5[0] == 0.0 # verify this is the singular case.

▶ What you'll see: variance is zero, but the denominator is a small positive number.

In [ ]:
xhat_b5 = (X_b5 - X_b5.mean(axis=0)) / den_b5 # normalize safely despite zero variance.
print("normalized constant feature:", xhat_b5.ravel()) # inspect the safe result.
plt.figure(figsize=(4, 3)) # create a simple plot.
plt.plot(xhat_b5.ravel(), marker="o", color="seagreen") # show all normalized values.
plt.title("Basic 5: epsilon avoids division by zero") # title the figure.
plt.ylim(-1, 1) # keep zero visible.
plt.show() # display the chart.

▶ What you'll see: all outputs are zero because the centered constant feature has no deviations.

👀 Takeaway: epsilon guarantees stable arithmetic, but it does not create variation where none exists.

### Basic 6 — Apply gamma scaling

**Goal.** Multiply normalized activations by γ, because the model may need a feature's standardized variation amplified or damped. We build it in 2 steps.

In [ ]:
xhat_b6 = np.array([[-1.0, -1.0], [0.0, 0.0], [1.0, 1.0]]) # create normalized toy activations.
gamma_b6 = np.array([2.0, 0.5]) # scale the first feature up and the second feature down.
scaled_b6 = gamma_b6 * xhat_b6 # broadcast gamma over examples.
print("scaled activations:\n", scaled_b6) # inspect the effect of learned scales.
assert np.allclose(scaled_b6.var(axis=0), [8/3, 1/6]) # verify variance scales by gamma squared.

▶ What you'll see: the first feature spreads four times more than the original, while the second shrinks.

In [ ]:
plt.figure(figsize=(4, 3)) # create a scale-comparison plot.
plt.plot(xhat_b6[:, 0], marker="o", label="before") # original standardized feature.
plt.plot(scaled_b6[:, 0], marker="o", label="after γ=2") # scaled feature.
plt.title("Basic 6: γ changes spread") # title the chart.
plt.legend() # show labels.
plt.show() # display the chart.

▶ What you'll see: γ=2 doubles distances from zero for feature 0.

👀 Takeaway: γ controls output variance after normalization.

### Basic 7 — Apply beta shifting

**Goal.** Add β after scaling, because the model may need a nonzero activation mean before the next nonlinearity. We build it in 2 steps.

In [ ]:
xhat_b7 = np.array([[-1.0, 0.0], [0.0, 1.0], [1.0, 2.0]]) # create normalized-like activations.
beta_b7 = np.array([3.0, -2.0]) # choose different learned offsets per feature.
shifted_b7 = xhat_b7 + beta_b7 # add beta by broadcasting.
print("shifted means:", shifted_b7.mean(axis=0)) # inspect new feature means.
assert np.allclose(shifted_b7.mean(axis=0), [3.0, -1.0]) # verify beta changes the mean.

▶ What you'll see: beta moves the output center without changing feature spread.

In [ ]:
plt.figure(figsize=(4, 3)) # create a before-after shift plot.
plt.plot(xhat_b7[:, 0], marker="o", label="before") # feature before beta.
plt.plot(shifted_b7[:, 0], marker="o", label="after β=3") # feature after beta.
plt.title("Basic 7: β changes center") # title the plot.
plt.legend() # show labels.
plt.show() # display the chart.

▶ What you'll see: the whole feature trace shifts upward by exactly 3.

👀 Takeaway: β controls output mean after normalization.

### Basic 8 — Full BatchNorm forward pass

**Goal.** Combine statistics, normalization, γ, and β, because the forward pass is one differentiable map used inside a network. We build it in 2 steps.

In [ ]:
X_b8 = np.array([[2., 10.], [4., 14.], [6., 18.]]) # create a small activation batch.
gamma_b8 = np.array([1.5, 0.5]) # set learned scales.
beta_b8 = np.array([0.0, 2.0]) # set learned shifts.
Y_b8, Xhat_b8, mu_b8, var_b8 = batch_norm_forward(X_b8, gamma_b8, beta_b8) # run the helper forward pass.
print("mu:", mu_b8, "var:", np.round(var_b8, 3)) # inspect batch statistics.

▶ What you'll see: the helper exposes the same statistics used internally by BatchNorm.

In [ ]:
print("output:\n", np.round(Y_b8, 3)) # inspect final BatchNorm outputs.
print("output mean:", np.round(Y_b8.mean(axis=0), 3)) # check beta controls the center.
assert np.allclose(np.round(Y_b8.mean(axis=0), 3), [0.0, 2.0]) # verify mean equals beta in this symmetric batch.
plt.figure(figsize=(4, 3)) # create output heatmap.
plt.imshow(Y_b8, cmap="coolwarm", aspect="auto") # visualize BatchNorm outputs.
plt.colorbar(label="y") # add output scale.
plt.title("Basic 8: full BN output") # title the plot.
plt.show() # display the heatmap.

▶ What you'll see: the output is centered around β and spread according to γ.

👀 Takeaway: BatchNorm is normalization followed by a learned affine transformation.

### Basic 9 — Update a running mean

**Goal.** Smooth batch means into a running mean, because inference should not depend on one example's statistics. We build it in 2 steps.

In [ ]:
running_b9 = np.array([0.0, 0.0]) # initialize running mean before training batches are seen.
batch_mu_b9 = np.array([4.0, 10.0]) # current minibatch mean.
momentum_b9 = 0.9 # keep most of the old estimate.
updated_b9 = running_update(running_b9, batch_mu_b9, momentum_b9) # apply exponential smoothing.
print("updated running mean:", updated_b9) # inspect the partial move toward batch_mu.
assert np.allclose(updated_b9, [0.4, 1.0]) # verify 10% of the batch statistic was added.

▶ What you'll see: the running mean moves only partway toward the current minibatch.

In [ ]:
plt.figure(figsize=(4, 3)) # create a comparison bar chart.
plt.bar(["old f0", "batch f0", "new f0"], [running_b9[0], batch_mu_b9[0], updated_b9[0]], color=["gray", "orange", "teal"]) # show smoothing on feature 0.
plt.title("Basic 9: running mean smoothing") # title the chart.
plt.ylabel("mean estimate") # label the axis.
plt.show() # display the plot.

▶ What you'll see: the new estimate is much closer to the old value than to the current batch.

👀 Takeaway: momentum makes inference statistics stable by averaging across many training batches.

### Basic 10 — Normalize an inference example

**Goal.** Use stored running statistics at inference, because deployment inputs should be transformed deterministically. We build it in 2 steps.

In [ ]:
x_b10 = np.array([[5.0, 12.0]]) # one inference example.
run_mu_b10 = np.array([3.0, 10.0]) # stored mean from training.
run_var_b10 = np.array([4.0, 16.0]) # stored variance from training.
gamma_b10 = np.array([1.0, 2.0]) # learned scale.
beta_b10 = np.array([0.0, -1.0]) # learned shift.
xhat_b10 = (x_b10 - run_mu_b10) / np.sqrt(run_var_b10 + 1e-5) # normalize with running statistics.
print("inference xhat:", np.round(xhat_b10, 3)) # inspect deterministic standardized values.
assert np.allclose(np.round(xhat_b10, 3), [[1.0, 0.5]]) # verify concrete normalized values.

▶ What you'll see: the single example is normalized with training-time estimates, not with its own mean.

In [ ]:
y_b10 = gamma_b10 * xhat_b10 + beta_b10 # apply learned affine parameters.
print("inference output:", np.round(y_b10, 3)) # inspect final deterministic output.
plt.figure(figsize=(4, 3)) # create a small output chart.
plt.bar(["feature 0", "feature 1"], y_b10.ravel(), color="purple") # show inference outputs.
plt.title("Basic 10: inference BatchNorm output") # title the chart.
plt.ylabel("y") # label output scale.
plt.show() # display the plot.

▶ What you'll see: feature 1 becomes 0 because γ·0.5 + β = 2·0.5 − 1.

👀 Takeaway: inference BatchNorm uses learned parameters plus stored running statistics.

## 🟡 Easy

### Easy 1 — Compare raw and normalized feature scales

**Goal.** Show how BatchNorm makes mismatched feature scales comparable, because later weights should not fight huge activation units. We build it in 3 steps.

In [ ]:
X_e1 = np.column_stack([np.linspace(0, 100, 20), np.linspace(-1, 1, 20)]) # create two features with very different units.
print("raw stds:", np.round(X_e1.std(axis=0), 3)) # inspect the original scale mismatch.
assert X_e1.std(axis=0)[0] > 25 * X_e1.std(axis=0)[1] # verify the first feature dominates raw scale.

▶ What you'll see: feature 0 has a much larger standard deviation than feature 1.

In [ ]:
Xhat_e1 = (X_e1 - X_e1.mean(axis=0)) / np.sqrt(X_e1.var(axis=0) + 1e-5) # normalize columns like BatchNorm.
print("normalized stds:", np.round(Xhat_e1.std(axis=0), 3)) # inspect the corrected scale.
assert np.allclose(np.round(Xhat_e1.std(axis=0), 3), [1.0, 1.0]) # verify unit standard deviation.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
plt.figure(figsize=(5, 3)) # create a side-by-side scale plot.
plt.plot(X_e1[:, 0], label="raw f0") # large-scale feature.
plt.plot(X_e1[:, 1], label="raw f1") # small-scale feature.
plt.plot(Xhat_e1[:, 0], linestyle="--", label="normalized f0") # standardized feature.
plt.plot(Xhat_e1[:, 1], linestyle="--", label="normalized f1") # standardized feature.
plt.title("Easy 1: raw units vs normalized units") # title the comparison.
plt.legend() # show curve labels.
plt.show() # display the plot.

▶ What you'll see: raw curves live on incompatible scales, while dashed normalized curves share a scale.

👀 Takeaway: BatchNorm makes feature magnitudes comparable before the next parameter update.

### Easy 2 — Show γ and β can undo normalization

**Goal.** Reconstruct a raw feature from its normalized version, because γ and β preserve expressivity when set to the original standard deviation and mean. We build it in 3 steps.

In [ ]:
x_e2 = np.array([[2.0], [4.0], [6.0], [8.0]]) # create one feature column.
mu_e2 = x_e2.mean(axis=0) # compute original mean.
std_e2 = x_e2.std(axis=0) # compute original standard deviation.
xhat_e2 = (x_e2 - mu_e2) / std_e2 # normalize without epsilon for the exact reconstruction demo.
print("mu/std:", mu_e2[0], round(std_e2[0], 3)) # inspect affine reconstruction parameters.

▶ What you'll see: the original feature can be described by one center and one scale.

In [ ]:
gamma_e2 = std_e2 # choose gamma as the original standard deviation.
beta_e2 = mu_e2 # choose beta as the original mean.
recon_e2 = gamma_e2 * xhat_e2 + beta_e2 # undo the normalization exactly.
print("reconstructed:", recon_e2.ravel()) # inspect recovered raw values.
assert np.allclose(recon_e2, x_e2) # verify gamma and beta can restore the original feature.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
plt.figure(figsize=(4, 3)) # create reconstruction plot.
plt.plot(x_e2.ravel(), marker="o", label="original") # original raw feature.
plt.plot(recon_e2.ravel(), marker="x", linestyle="--", label="γ xhat + β") # reconstructed feature.
plt.title("Easy 2: affine parameters restore scale") # title the figure.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the two traces overlap exactly.

👀 Takeaway: learned γ and β mean BatchNorm constrains optimization without permanently limiting representation.

### Easy 3 — Momentum controls running-stat speed

**Goal.** Compare different momentum values, because running statistics can react quickly or slowly to changing batches. We build it in 3 steps.

In [ ]:
batch_means_e3 = np.array([0.0, 10.0, 10.0, 10.0]) # simulate a sudden shift in activation mean.
momenta_e3 = [0.5, 0.9] # compare fast and slow smoothing.
print("batch means:", batch_means_e3) # inspect the statistic stream.

▶ What you'll see: the mean jumps from 0 to 10 and stays there.

In [ ]:
histories_e3 = [] # store running means for each momentum.
for m_e3 in momenta_e3: # loop over smoothing strengths.
    run_e3 = 0.0 # start with zero running mean.
    hist_e3 = [] # store this curve.
    for b_e3 in batch_means_e3: # process each batch statistic.
        run_e3 = m_e3 * run_e3 + (1 - m_e3) * b_e3 # exponential moving average update.
        hist_e3.append(run_e3) # record the new estimate.
    histories_e3.append(hist_e3) # save this momentum curve.
print("m=0.5:", np.round(histories_e3[0], 3), "m=0.9:", np.round(histories_e3[1], 3)) # inspect response speeds.
assert histories_e3[0][-1] > histories_e3[1][-1] # lower momentum reacts faster.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
plt.figure(figsize=(5, 3)) # create a momentum comparison plot.
plt.plot(histories_e3[0], marker="o", label="momentum 0.5") # fast-running estimate.
plt.plot(histories_e3[1], marker="o", label="momentum 0.9") # slow-running estimate.
plt.axhline(10, color="gray", linestyle="--", label="new batch mean") # target statistic.
plt.title("Easy 3: momentum changes running-stat lag") # title the figure.
plt.xlabel("batch index") # label time axis.
plt.ylabel("running mean") # label estimate axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: momentum 0.5 catches up faster, while momentum 0.9 changes slowly.

👀 Takeaway: BatchNorm momentum trades quick adaptation against smooth inference statistics.

### Easy 4 — Mini-batch noise acts like regularization

**Goal.** Observe that different minibatches normalize the same value differently, because batch statistics inject small training-time noise. We build it in 3 steps.

In [ ]:
rng_e4 = np.random.default_rng(4) # create a reproducible generator.
pop_e4 = rng_e4.normal(loc=5.0, scale=2.0, size=200) # simulate a population of activations.
probe_e4 = 5.0 # fixed activation value to normalize under different batches.
print("population mean/std:", round(pop_e4.mean(), 3), round(pop_e4.std(), 3)) # inspect global reference.

▶ What you'll see: the full population has a stable center and spread.

In [ ]:
zs_e4 = [] # store normalized probe values under different batch samples.
for k_e4 in range(20): # draw several training minibatches.
    batch_e4 = rng_e4.choice(pop_e4, size=8, replace=False) # small batch creates noisy statistics.
    z_e4 = (probe_e4 - batch_e4.mean()) / np.sqrt(batch_e4.var() + 1e-5) # normalize the same probe with this batch.
    zs_e4.append(z_e4) # record the result.
zs_e4 = np.array(zs_e4) # convert to array for summaries.
print("probe z range:", round(zs_e4.min(), 3), "to", round(zs_e4.max(), 3)) # inspect training-time variation.
assert zs_e4.max() - zs_e4.min() > 0.5 # verify noticeable batch noise.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
plt.figure(figsize=(5, 3)) # create a histogram of probe z-scores.
plt.hist(zs_e4, bins=8, color="slateblue", edgecolor="white") # show variation from minibatch statistics.
plt.title("Easy 4: same activation, different batch stats") # title the figure.
plt.xlabel("normalized probe value") # label x-axis.
plt.ylabel("count") # label y-axis.
plt.show() # display the histogram.

▶ What you'll see: the same raw activation maps to a range of normalized values across minibatches.

👀 Takeaway: minibatch statistics add noise that can regularize training, but too much noise hurts tiny batches.

### Easy 5 — Check where BatchNorm sits around ReLU

**Goal.** Compare ReLU before and after normalization, because centering before a nonlinearity changes how many activations stay alive. We build it in 3 steps.

In [ ]:
z_e5 = np.array([[-3.0], [-1.0], [1.0], [3.0], [5.0]]) # create shifted pre-activations.
relu_raw_e5 = np.maximum(0.0, z_e5) # apply ReLU directly to raw shifted values.
xhat_e5 = (z_e5 - z_e5.mean(axis=0)) / np.sqrt(z_e5.var(axis=0) + 1e-5) # normalize before ReLU.
relu_bn_e5 = np.maximum(0.0, xhat_e5) # apply ReLU after BatchNorm-style centering.
print("raw positive count:", int((relu_raw_e5 > 0).sum()), "BN positive count:", int((relu_bn_e5 > 0).sum())) # inspect alive activations.

▶ What you'll see: normalization changes which values are above the ReLU threshold.

In [ ]:
print("xhat:", np.round(xhat_e5.ravel(), 3)) # inspect centered pre-ReLU values.
assert abs(float(xhat_e5.mean())) < 1e-12 # verify centering before ReLU.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
plt.figure(figsize=(5, 3)) # create a before-after ReLU plot.
plt.plot(relu_raw_e5.ravel(), marker="o", label="ReLU(raw z)") # raw ReLU outputs.
plt.plot(relu_bn_e5.ravel(), marker="o", label="ReLU(BN z)") # normalized ReLU outputs.
plt.title("Easy 5: centering changes ReLU gates") # title the figure.
plt.xlabel("example index") # label examples.
plt.ylabel("activation after ReLU") # label output.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: BatchNorm recenters the pre-activations, changing the ReLU gate pattern and output scale.

👀 Takeaway: BatchNorm is usually placed before the nonlinearity so the gate sees controlled pre-activation statistics.

## 🔴 Advanced

### Advanced 1 — Train γ and β for a target mean and variance

**Goal.** Learn affine BatchNorm parameters with gradient descent, because γ and β are ordinary trainable parameters. We build it in 4 steps.

In [ ]:
xhat_a1 = np.linspace(-1.5, 1.5, 9).reshape(-1, 1) # fixed normalized activations with mean 0.
target_a1 = 2.0 * xhat_a1 + 3.0 # desired post-BN output has scale 2 and shift 3.
gamma_a1 = np.array([0.0]) # start with wrong scale.
beta_a1 = np.array([0.0]) # start with wrong shift.
print("target mean/std:", round(float(target_a1.mean()), 3), round(float(target_a1.std()), 3)) # inspect desired output statistics.

▶ What you'll see: the target output is centered at 3 and has nonzero spread.

In [ ]:
losses_a1 = [] # store optimization loss.
for step_a1 in range(120): # run small gradient-descent updates.
    pred_a1 = gamma_a1 * xhat_a1 + beta_a1 # current affine BatchNorm output.
    err_a1 = pred_a1 - target_a1 # residual against desired output.
    grad_gamma_a1 = np.mean(2 * err_a1 * xhat_a1, axis=0) # derivative through y = gamma*xhat + beta.
    grad_beta_a1 = np.mean(2 * err_a1, axis=0) # derivative through beta shift.
    gamma_a1 -= 0.1 * grad_gamma_a1 # update scale parameter.
    beta_a1 -= 0.1 * grad_beta_a1 # update shift parameter.
    losses_a1.append(float(np.mean(err_a1 ** 2))) # record mean squared error.
print("learned gamma/beta:", np.round(gamma_a1, 3), np.round(beta_a1, 3)) # inspect learned affine parameters.
assert abs(gamma_a1[0] - 2.0) < 0.02 and abs(beta_a1[0] - 3.0) < 0.01 # verify convergence.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
final_a1 = gamma_a1 * xhat_a1 + beta_a1 # compute learned output.
print("final loss:", round(losses_a1[-1], 6)) # inspect final fit.
assert losses_a1[-1] < losses_a1[0] # verify optimization improved the affine fit.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
plt.figure(figsize=(5, 3)) # create the learning curve.
plt.plot(losses_a1, color="purple") # plot MSE over updates.
plt.title("Advanced 1: γ and β learn by gradients") # title the figure.
plt.xlabel("step") # label optimization step.
plt.ylabel("MSE") # label loss.
plt.show() # display the curve.

▶ What you'll see: the loss falls as γ approaches 2 and β approaches 3.

👀 Takeaway: BatchNorm's affine parameters are learned with the same gradient machinery as weights.

### Advanced 2 — Derive the backward pass numerically

**Goal.** Compute gradients through normalization, because the mean and variance couple all examples in a batch. We build it in 4 steps.

In [ ]:
x_a2 = np.array([[1.0], [2.0], [4.0]]) # one feature over three examples.
gamma_a2 = np.array([1.5]) # scale parameter.
beta_a2 = np.array([0.2]) # shift parameter.
dout_a2 = np.array([[1.0], [-2.0], [3.0]]) # upstream gradient from the next layer.
print("x:", x_a2.ravel(), "dout:", dout_a2.ravel()) # inspect backward inputs.

▶ What you'll see: one upstream gradient per example flows into a shared normalized feature.

In [ ]:
mu_a2 = x_a2.mean(axis=0) # forward mean.
var_a2 = x_a2.var(axis=0) # forward variance.
xhat_a2 = (x_a2 - mu_a2) / np.sqrt(var_a2 + 1e-5) # forward normalized values.
dgamma_a2 = np.sum(dout_a2 * xhat_a2, axis=0) # gradient wrt gamma.
dbeta_a2 = np.sum(dout_a2, axis=0) # gradient wrt beta.
print("dgamma/dbeta:", np.round(dgamma_a2, 3), dbeta_a2) # inspect affine parameter gradients.
assert dbeta_a2[0] == 2.0 # verify beta gradient is sum of upstream gradients.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
N_a2 = x_a2.shape[0] # batch size.
std_inv_a2 = 1.0 / np.sqrt(var_a2 + 1e-5) # inverse standard deviation.
dxhat_a2 = dout_a2 * gamma_a2 # upstream gradient into xhat.
dx_a2 = (1.0 / N_a2) * std_inv_a2 * (N_a2 * dxhat_a2 - np.sum(dxhat_a2, axis=0) - xhat_a2 * np.sum(dxhat_a2 * xhat_a2, axis=0)) # compact BatchNorm backward formula.
print("dx:", np.round(dx_a2.ravel(), 6)) # inspect gradients wrt input examples.
assert abs(float(dx_a2.sum())) < 1e-10 # centered normalization makes input gradients sum to zero.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
plt.figure(figsize=(5, 3)) # create gradient bar plot.
plt.bar(["x0", "x1", "x2"], dx_a2.ravel(), color="crimson") # show coupled input gradients.
plt.axhline(0, color="black", linewidth=0.8) # mark zero.
plt.title("Advanced 2: BN backward couples examples") # title the figure.
plt.ylabel("dx") # label gradient axis.
plt.show() # display the plot.

▶ What you'll see: gradients can be positive or negative, and they sum to approximately zero.

👀 Takeaway: BatchNorm backward is not elementwise; every example affects the batch mean and variance used by the others.

### Advanced 3 — Batch size changes statistic reliability

**Goal.** Measure variance-estimate noise across batch sizes, because BatchNorm depends on minibatch statistics during training. We build it in 3 steps.

In [ ]:
rng_a3 = np.random.default_rng(3) # reproducible simulation.
pop_a3 = rng_a3.normal(loc=0.0, scale=2.0, size=5000) # large population with true variance near 4.
batch_sizes_a3 = np.array([2, 4, 16, 64]) # compare tiny to larger batches.
print("population variance:", round(float(pop_a3.var()), 3)) # inspect reference variance.

▶ What you'll see: the population variance is close to 4.

In [ ]:
var_std_a3 = [] # store standard deviation of variance estimates.
for bs_a3 in batch_sizes_a3: # loop over batch sizes.
    estimates_a3 = [] # collect many variance estimates.
    for _ in range(300): # repeat sampling.
        sample_a3 = rng_a3.choice(pop_a3, size=int(bs_a3), replace=False) # draw one minibatch.
        estimates_a3.append(sample_a3.var()) # record its variance estimate.
    var_std_a3.append(np.std(estimates_a3)) # summarize estimator noise.
print("std of variance estimates:", np.round(var_std_a3, 3)) # inspect reliability by batch size.
assert var_std_a3[0] > var_std_a3[-1] # verify larger batches are more reliable.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
plt.figure(figsize=(5, 3)) # create reliability curve.
plt.plot(batch_sizes_a3, var_std_a3, marker="o", color="darkorange") # plot estimator noise versus batch size.
plt.title("Advanced 3: larger batches give steadier variance") # title the plot.
plt.xlabel("batch size") # label x-axis.
plt.ylabel("std of variance estimate") # label y-axis.
plt.show() # display the curve.

▶ What you'll see: variance estimates are much noisier for batch size 2 than for batch size 64.

👀 Takeaway: BatchNorm's behavior depends on having enough examples to estimate useful statistics.

### Advanced 4 — Simulate a distribution shift at inference

**Goal.** Compare training-batch normalization with running-stat inference under shifted data, because stale running statistics can miscalibrate deployment activations. We build it in 4 steps.

In [ ]:
run_mu_a4 = np.array([0.0]) # stored training mean.
run_var_a4 = np.array([1.0]) # stored training variance.
gamma_a4 = np.array([1.0]) # identity learned scale.
beta_a4 = np.array([0.0]) # identity learned shift.
X_shift_a4 = np.array([[3.0], [4.0], [5.0], [6.0]]) # inference batch shifted away from training distribution.
print("shifted batch mean:", X_shift_a4.mean()) # inspect deployment shift.

▶ What you'll see: inference activations are centered far from the stored training mean.

In [ ]:
y_infer_a4 = gamma_a4 * ((X_shift_a4 - run_mu_a4) / np.sqrt(run_var_a4 + 1e-5)) + beta_a4 # inference uses stored stats.
y_batch_a4 = gamma_a4 * ((X_shift_a4 - X_shift_a4.mean(axis=0)) / np.sqrt(X_shift_a4.var(axis=0) + 1e-5)) + beta_a4 # training-style batch stats.
print("inference mean:", round(float(y_infer_a4.mean()), 3), "batch-normalized mean:", round(float(y_batch_a4.mean()), 3)) # compare centers.
assert float(y_infer_a4.mean()) > 4.0 and abs(float(y_batch_a4.mean())) < 1e-12 # verify shift effect.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
print("first inference output:", round(float(y_infer_a4[0, 0]), 3), "first batch-stat output:", round(float(y_batch_a4[0, 0]), 3)) # inspect one example.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
plt.figure(figsize=(5, 3)) # create shift comparison plot.
plt.plot(y_infer_a4.ravel(), marker="o", label="using running stats") # deployment behavior.
plt.plot(y_batch_a4.ravel(), marker="o", label="using current batch stats") # training-like behavior.
plt.axhline(0, color="black", linewidth=0.8) # mark zero center.
plt.title("Advanced 4: stale running stats under shift") # title the figure.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: running-stat outputs remain strongly positive because the stored mean is stale.

👀 Takeaway: BatchNorm inference is deterministic, but it assumes running statistics still match deployment data.

### Advanced 5 — Inspect memory and parameter cost

**Goal.** Count BatchNorm parameters and cached activations, because normalization is mathematically small but still has training-time memory cost. We build it in 3 steps.

In [ ]:
batch_a5 = 32 # number of examples in a minibatch.
features_a5 = 128 # number of feature channels.
bytes_float_a5 = 4 # float32 bytes.
params_a5 = 2 * features_a5 # gamma and beta per feature.
activation_bytes_a5 = batch_a5 * features_a5 * bytes_float_a5 # one activation matrix cache size.
print("parameters γ+β:", params_a5) # inspect trainable parameter count.
print("one activation cache KB:", activation_bytes_a5 / 1024) # inspect memory scale.
assert params_a5 == 256 and activation_bytes_a5 / 1024 == 16.0 # verify concrete bookkeeping.

▶ What you'll see: BatchNorm adds few parameters but caches activations proportional to batch×features.

In [ ]:
cached_arrays_a5 = 3 # xhat, mean/variance-related pieces, and upstream-needed activations in a simple implementation.
approx_cache_kb_a5 = cached_arrays_a5 * activation_bytes_a5 / 1024 # rough training cache footprint.
print("rough cache KB for 3 activation-sized arrays:", approx_cache_kb_a5) # inspect training memory.
assert approx_cache_kb_a5 == 48.0 # verify the arithmetic.

▶ What you'll see: the printed values or plot expose the intermediate BatchNorm quantity for inspection.

In [ ]:
plt.figure(figsize=(5, 3)) # create bookkeeping chart.
plt.bar(["γ+β params", "one cache KB", "rough 3-cache KB"], [params_a5, activation_bytes_a5 / 1024, approx_cache_kb_a5], color=["teal", "orange", "crimson"]) # compare counts on one axis.
plt.title("Advanced 5: BN parameter and memory bookkeeping") # title the chart.
plt.ylabel("count or KB") # label mixed bookkeeping units.
plt.xticks(rotation=15) # keep labels readable.
plt.show() # display the plot.

▶ What you'll see: parameters are tiny compared with activation storage once batch and feature counts grow.

👀 Takeaway: BatchNorm's trainable parameter cost is small, but its cached activation statistics matter in large networks.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Batch normalization standardizes activations with batch statistics, then learns how much scale and shift to restore.

Minibatches estimate full-data quantities. BatchNorm makes that estimation explicit by computing a mean and variance during training, then reusing running training statistics at evaluation time. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, standardize, predict, and return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def one_hot(y, k):
    out = np.zeros((len(y), k))
    out[np.arange(len(y)), y.astype(int)] = 1.0
    return out


def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def random_relu_features(X, seed=0, width=24):
    rng = np.random.default_rng(seed + X.shape[1])
    W = rng.normal(0.0, 1.0 / np.sqrt(max(1, X.shape[1])), size=(X.shape[1], width))
    b = rng.normal(0.0, 0.15, size=width)
    H = np.maximum(0.0, X @ W + b)
    pair = X[:, :1] * X[:, 1:2] if X.shape[1] >= 2 else X
    return np.hstack([X, X * X, pair, H])


def batch_norm_fit(H, eps=1e-5):
    mu = H.mean(axis=0, keepdims=True)
    var = H.var(axis=0, keepdims=True)
    Z = (H - mu) / np.sqrt(var + eps)
    return Z, (mu, var, eps)


def batch_norm_apply(H, params):
    mu, var, eps = params
    return (H - mu) / np.sqrt(var + eps)


def layer_norm(H, eps=1e-5):
    mu = H.mean(axis=1, keepdims=True)
    var = H.var(axis=1, keepdims=True)
    return (H - mu) / np.sqrt(var + eps)


def group_norm(H, groups=4, eps=1e-5):
    usable = (H.shape[1] // groups) * groups
    head = H[:, :usable].reshape(H.shape[0], groups, -1)
    mu = head.mean(axis=2, keepdims=True)
    var = head.var(axis=2, keepdims=True)
    normed = ((head - mu) / np.sqrt(var + eps)).reshape(H.shape[0], usable)
    if usable == H.shape[1]:
        return normed
    return np.hstack([normed, H[:, usable:]])


def instance_norm(H, eps=1e-5):
    usable = (H.shape[1] // 8) * 8
    head = H[:, :usable].reshape(H.shape[0], 8, -1)
    mu = head.mean(axis=2, keepdims=True)
    var = head.var(axis=2, keepdims=True)
    normed = ((head - mu) / np.sqrt(var + eps)).reshape(H.shape[0], usable)
    if usable == H.shape[1]:
        return normed
    return np.hstack([normed, H[:, usable:]])


def deep_random_features(X, depth=4, scale=1.0, residual=False, seed=0):
    H = random_relu_features(X, seed=seed, width=20)
    rng = np.random.default_rng(seed + 100 + H.shape[1])
    for _ in range(depth):
        W = rng.normal(0.0, scale / np.sqrt(H.shape[1]), size=(H.shape[1], H.shape[1]))
        F = np.maximum(0.0, H @ W)
        if residual:
            H = H + 0.35 * F
        else:
            H = F
    return H


def transform_pair(x_tr, x_te, mode="plain", seed=0, scale=1.0, residual=False):
    Htr = random_relu_features(x_tr, seed=seed)
    Hte = random_relu_features(x_te, seed=seed)
    if mode == "batchnorm":
        Htr, params = batch_norm_fit(Htr)
        Hte = batch_norm_apply(Hte, params)
    if mode == "test_batchnorm_wrong":
        Htr, params = batch_norm_fit(Htr)
        Hte, _ = batch_norm_fit(Hte)
    if mode == "layernorm":
        Htr = layer_norm(Htr)
        Hte = layer_norm(Hte)
    if mode == "groupnorm":
        Htr = group_norm(Htr)
        Hte = group_norm(Hte)
    if mode == "instancenorm":
        Htr = instance_norm(Htr)
        Hte = instance_norm(Hte)
    if mode == "deep":
        Htr = deep_random_features(x_tr, depth=5, scale=scale, residual=residual, seed=seed)
        Hte = deep_random_features(x_te, depth=5, scale=scale, residual=residual, seed=seed)
    return Htr, Hte


def train_softmax_classifier(x_tr, y_tr, x_te, epsilon=0.0, epochs=40, lr=0.2, clip=None, schedule="constant", transform="plain", seed=0, scale=1.0, residual=False):
    Htr, Hte = transform_pair(x_tr, x_te, mode=transform, seed=seed, scale=scale, residual=residual)
    if len(y_tr) > 700:
        rng_sub = np.random.default_rng(seed + 700)
        idx_sub = rng_sub.choice(len(y_tr), size=700, replace=False)
        Htr = Htr[idx_sub]
        y_tr = y_tr[idx_sub]
    k = int(y_tr.max()) + 1
    Y = one_hot(y_tr, k)
    targets = (1.0 - epsilon) * Y + epsilon / k
    rng = np.random.default_rng(seed + 10)
    W = rng.normal(0.0, 0.01, size=(Htr.shape[1], k))
    b = np.zeros(k)
    losses = []
    grad_norms = []
    for epoch in range(epochs):
        eta = lr_value(schedule, epoch, epochs, lr)
        P = softmax(Htr @ W + b)
        loss = -np.mean(np.sum(targets * np.log(P + 1e-12), axis=1))
        G = (P - targets) / len(y_tr)
        dW = Htr.T @ G
        db = G.sum(axis=0)
        norm = float(np.sqrt(np.sum(dW * dW) + np.sum(db * db)))
        if clip is not None:
            factor = min(1.0, clip / (norm + 1e-12))
            dW = dW * factor
            db = db * factor
        W = W - eta * dW
        b = b - eta * db
        losses.append(float(loss))
        grad_norms.append(norm)
    preds = np.argmax(Hte @ W + b, axis=1)
    return preds, losses, grad_norms


def lr_value(schedule, epoch, epochs, base):
    if schedule == "constant":
        return base
    if schedule == "step":
        return base if epoch < epochs // 2 else base * 0.2
    if schedule == "cosine":
        return 0.02 * base + 0.5 * (base - 0.02 * base) * (1.0 + math.cos(math.pi * epoch / max(1, epochs - 1)))
    if schedule == "warmup_cosine":
        warm = max(2, epochs // 5)
        if epoch < warm:
            return base * (epoch + 1) / warm
        span = max(1, epochs - warm - 1)
        t = epoch - warm
        return 0.02 * base + 0.5 * (base - 0.02 * base) * (1.0 + math.cos(math.pi * t / span))
    if schedule == "onecycle":
        half = max(1, epochs // 2)
        if epoch < half:
            return base * (0.2 + 1.8 * epoch / half)
        return base * (2.0 - 1.8 * (epoch - half) / max(1, epochs - half))
    return base


def component_accuracy(name, X, y, **kwargs):
    def build(x_tr, y_tr, x_te):
        preds, _, _ = train_softmax_classifier(x_tr, y_tr, x_te, **kwargs)
        return preds
    return clf_accuracy(build, X, y)


def fit_softmax_on_features(Htr, y_tr, Hte, epochs=40, lr=0.3, seed=0):
    if len(y_tr) > 700:
        rng_sub = np.random.default_rng(seed + 701)
        idx_sub = rng_sub.choice(len(y_tr), size=700, replace=False)
        Htr = Htr[idx_sub]
        y_tr = y_tr[idx_sub]
    k = int(y_tr.max()) + 1
    Y = one_hot(y_tr, k)
    rng = np.random.default_rng(seed + Htr.shape[1])
    W = rng.normal(0.0, 0.01, size=(Htr.shape[1], k))
    b = np.zeros(k)
    for epoch in range(epochs):
        P = softmax(Htr @ W + b)
        G = (P - Y) / len(y_tr)
        dW = Htr.T @ G
        db = G.sum(axis=0)
        W = W - lr * dW
        b = b - lr * db
    return np.argmax(Hte @ W + b, axis=1)


def logistic_accuracy_for_features(X, y, mode="plain", scale=1.0, residual=False, seed=0):
    def build(x_tr, y_tr, x_te):
        Htr, Hte = transform_pair(x_tr, x_te, mode=mode, seed=seed, scale=scale, residual=residual)
        return fit_softmax_on_features(Htr, y_tr, Hte, epochs=40, lr=0.35, seed=seed)
    return clf_accuracy(build, X, y)


def ladder_preview(rungs):
    for name, X, y in rungs:
        classes = np.unique(y)
        print(f"{name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")
    print("First D1 sample:", rungs[0][1][0].tolist(), "label=", int(rungs[0][2][0]))


def print_metric_table(rows, header="rung metric"):
    print(header)
    for name, metric in rows:
        print(f"{name:36s} {metric:.3f}")


def split_for_demo(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def plot_ladder_results(rungs, metrics, title, artifact_fn=None):
    fig, axes = plt.subplots(1, 5, figsize=(16, 3))
    for ax, (name, X, y) in zip(axes, rungs):
        if artifact_fn is None:
            if X.shape[1] == 64:
                ax.imshow(X[0].reshape(8, 8), cmap="gray")
            else:
                ax.scatter(X[:, 0], X[:, 1], c=y, cmap="tab10", s=12)
        else:
            artifact_fn(ax, name, X, y)
        ax.set_title(name.split()[0])
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title + " artifacts")
    plt.show()

    plt.figure(figsize=(6, 3))
    plt.plot(range(1, 6), metrics, marker="o")
    plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
    plt.ylim(0.0, 1.05)
    plt.ylabel("held-out accuracy")
    plt.title(title + " summary")
    plt.grid(True, alpha=0.3)
    plt.show()

## The concept, built once

The lesson formula is $\hat x_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}},\ y_i=\gamma\hat x_i+\beta$. On the four D1 XOR first-coordinate activations $[0,1,0,1]$, the batch mean is $0.5$ and variance is $0.25$. The normalized values should be approximately $[-0.99998,0.99998,-0.99998,0.99998]$.

In [ ]:
activations = np.array([0.0, 1.0, 0.0, 1.0])
mu_b = activations.mean()
var_b = activations.var()
eps = 1e-5
normalized = (activations - mu_b) / np.sqrt(var_b + eps)
expected = np.array([-0.99998, 0.99998, -0.99998, 0.99998])
print("mu_B=", mu_b, "var_B=", var_b)
print("batch-normalized:", normalized)
assert np.allclose(normalized, expected, atol=1e-5)

This helper is the reusable method for the rest of the notebook. It keeps the model and ladder fixed, then varies only this topic's component.

In [ ]:
print('Reusable component method is available in the setup cell and verified above.')

## The dataset ladder

We use the shared F5 classification ladder: D1 XOR, D2 blobs, D3 noisy moons, D4 real sklearn digits, and D5 noisy digits. The same accuracy wrapper and model family run on every rung.

In [ ]:
rungs = clf_digits_ladder()
ladder_preview(rungs)

## Run the same method across D1–D5

The table reports one held-out accuracy per rung while the component-specific sweep is printed for auditability.

In [ ]:
rungs = clf_digits_ladder()
rows = []
for rung_id, (name, X, y) in enumerate(rungs):
    off = logistic_accuracy_for_features(X, y, mode="plain", seed=30 + rung_id)
    on = logistic_accuracy_for_features(X, y, mode="batchnorm", seed=30 + rung_id)
    rows.append((name, on))
    print(name, "BN off/on", round(off, 3), round(on, 3))
metrics = [metric for _, metric in rows]
print_metric_table(rows, "BatchNorm-on accuracy")

## Results visualization

The closing figure has two parts: a small multiple showing each rung's data artifact and a summary curve of the selected metric from D1 to D5.

In [ ]:
plot_ladder_results(rungs, metrics, '6.15 Batch normalization')

## Pitfall on D5

Ignoring train/eval mode leaks test-batch statistics. On D5, use training BatchNorm statistics for evaluation rather than recomputing from the test batch.

In [ ]:
name, X, y = clf_digits_ladder()[-1]
right = logistic_accuracy_for_features(X, y, mode="batchnorm", seed=77)
wrong = logistic_accuracy_for_features(X, y, mode="test_batchnorm_wrong", seed=77)
print("D5 eval with training stats:", round(right, 3))
print("D5 eval recomputing test-batch stats:", round(wrong, 3))
print("Fix: store training mean/variance and apply them during evaluation.")

## Evaluate it

- Metric: held-out accuracy from `clf_accuracy`; compare to a no-skill majority-class or plain-feature baseline.
- Sanity check: D1 should be inspectable and every probability target should sum to one when probabilities are used.
- Ablation: turn this topic's component off and verify the metric or diagnostic changes.
- Failure signal: unstable loss, axis mismatch, train/eval leakage, or D5 improvement without a matching diagnostic.

## Practice

1. Change one component value and rerun the D1 assertion plus the D1–D5 table.

2. Add a majority-class baseline to the summary curve.

3. On D5, print one extra diagnostic that would catch the named pitfall.